[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/geraldmc/irilab2026/blob/main/notebooks/r1/r1-q3/03_compare_and_interpret.ipynb)

# R1-Q3 Week 4 — Compare attributions to known stress biology

### This notebook produces a Strong / Mixed / Low Overlap verdict with the evidence underneath it.

This notebook compares the top-attributed genes from your SHAP analysis (Week 3) against curated lists of known stress-responsive genes — primarily Hakkak & Tohidfar 2026's consensus list. The result is the diagnostic for whether your classifier learned real stress biology or technical patterns in the data.

By the end of this notebook you will have:

- An overlap table comparing top-attributed genes against curated stress-responsive sets, with hypergeometric test results per stress class (`attribution_overlap.parquet`).
- The specific gene-level overlap detail showing which curated reference genes appeared in the top-N attributions per class (`overlap_genes.parquet`).
- An `attribution_comparison.json` recording the per-class overlap statistics, the test parameters, and the diagnostic verdict (Strong, Mixed, or Low Overlap), ready as the headline finding in your Week 4 paper and final presentation.

In [ ]:
# Cell 0 — probe
try:
    import irilab2026 as iri
    print(f"irilab2026 {iri.__version__} already installed.")
    print("If you want to pull the latest changes, run Cell 1 below.")
except ModuleNotFoundError:
    print("irilab2026 not installed — run Cell 1 below.")

In [ ]:
# Cell 1 — install
# First run in a fresh Colab session: run this cell.
# Later runs in the same session: skip this cell.
!pip install git+https://github.com/geraldmc/irilab2026.git@main

**A note about a possible restart prompt.** If Colab shows a "RESTART SESSION" banner after the install: click it, wait for the kernel to reconnect, then continue from the next cell. The install does not need to be run again. If no banner appears, ignore this and move on.

In [ ]:
# Cell 2 — mount and setup
# Mount Drive, set up irilab2026, point to the R1-Q3 output directory.
import irilab2026 as iri
iri.setup()

OUTPUT_DIR = iri.output_dir("r1_q3")
print(f"Output directory: {OUTPUT_DIR}")

### 1) Load Notebook 02's outputs and cross-check the contracts

The first thing this notebook does is read everything it needs from upstream: your precommitments from Notebook 00, the SHAP attribution outputs from Notebook 02, and the classifier summary from Notebook 01. It then cross-checks them against each other so any drift between notebooks surfaces here rather than further down.

The cross-checks fall into three buckets:

1. **Precommit structure.** The four top-level keys are present, the data source is `"GEO"`, the class count is `8` in both the places it appears, and the `artifact_like_definition` block has the sub-keys later sections will read.
2. **Cross-notebook consistency.** The attribution method recorded in `attribution_summary.json` matches what the precommit asked for, and the classifier referenced by Notebook 02 is the one Notebook 01 wrote.
3. **Subset selection.** Notebook 03 only processes subsets whose classifier passed the accuracy gate in Notebook 01. The list of passing subsets is read from `classifier_summary.json`.

If anything is off, this section fails loudly with a message that names which file is the source of the problem and what the expected value was.

In [ ]:
import json

# --- Resolve paths -----------------------------------------------------
# OUTPUT_DIR is set up in the preamble via iri.output_dir().
precommit_path            = OUTPUT_DIR / "precommit.json"
attribution_scores_path   = OUTPUT_DIR / "attribution_scores.parquet"
top_attributed_genes_path = OUTPUT_DIR / "top_attributed_genes.json"
attribution_summary_path  = OUTPUT_DIR / "attribution_summary.json"
classifier_summary_path   = OUTPUT_DIR / "classifier_summary.json"

# --- Defensive load of each input --------------------------------------
def _load_required(path, loader, hint):
    try:
        return loader(path)
    except FileNotFoundError:
        raise FileNotFoundError(
            f"\nCould not find {path.name}.\n"
            f"  Expected at: {path}\n\n"
            f"{hint}\n"
        ) from None

precommit            = _load_required(
    precommit_path, lambda p: json.loads(p.read_text()),
    "Re-run Notebook 00 end-to-end; its closeout writes precommit.json.",
)
attribution_scores   = _load_required(
    attribution_scores_path, pd.read_parquet,
    "Re-run Notebook 02 end-to-end; Section 6 writes attribution_scores.parquet.",
)
top_attributed_genes = _load_required(
    top_attributed_genes_path, lambda p: json.loads(p.read_text()),
    "Re-run Notebook 02 end-to-end; Section 6 writes top_attributed_genes.json.",
)
attribution_summary  = _load_required(
    attribution_summary_path, lambda p: json.loads(p.read_text()),
    "Re-run Notebook 02 end-to-end; Section 6 writes attribution_summary.json.",
)
classifier_summary   = _load_required(
    classifier_summary_path, lambda p: json.loads(p.read_text()),
    "Re-run Notebook 01 end-to-end; Section 4.3 writes classifier_summary.json.",
)

# --- Precommit: top-level keys -----------------------------------------
expected_top_keys = {
    "attribution_method",
    "reference_gene_set",
    "artifact_like_definition",
    "data_source_and_scope",
}
missing = expected_top_keys - set(precommit.keys())
assert not missing, (
    f"precommit.json is missing top-level keys: {sorted(missing)}.\n"
    f"Re-run Notebook 00 — its closeout writes the full schema."
)

# --- Precommit: data source and class count ----------------------------
data_scope = precommit["data_source_and_scope"]
assert data_scope["source"] == "GEO", (
    f"precommit.json: data_source_and_scope.source is "
    f"{data_scope['source']!r}; expected 'GEO'. The library serves GEO only."
)
assert data_scope["n_stress_classes"] == 8, (
    f"precommit.json: data_source_and_scope.n_stress_classes is "
    f"{data_scope['n_stress_classes']}; expected 8."
)

artifact_def = precommit["artifact_like_definition"]
assert artifact_def["n_classes"] == 8, (
    f"precommit.json: artifact_like_definition.n_classes is "
    f"{artifact_def['n_classes']}; expected 8."
)
assert artifact_def["n_classes"] == data_scope["n_stress_classes"], (
    "precommit.json: artifact_like_definition.n_classes does not match "
    "data_source_and_scope.n_stress_classes. Both should be 8."
)

# --- Precommit: artifact_like_definition sub-keys ----------------------
expected_artifact_subkeys = {
    "n_classes", "per_class", "overlap_clause", "metadata_clause",
    "combination", "rollup", "rationale",
}
missing_subkeys = expected_artifact_subkeys - set(artifact_def.keys())
assert not missing_subkeys, (
    f"precommit.json: artifact_like_definition is missing sub-keys: "
    f"{sorted(missing_subkeys)}. Re-run Notebook 00 Section 6."
)

# --- Cross-notebook: attribution method --------------------------------
precommit_method = precommit["attribution_method"]["method"]
nb02_method      = attribution_summary["setup"]["attribution_method"]
assert nb02_method == precommit_method, (
    f"Attribution method drift between notebooks:\n"
    f"  precommit.json:           {precommit_method!r}\n"
    f"  attribution_summary.json: {nb02_method!r}\n"
    f"Re-run Notebook 02 with the precommit-aligned method."
)

# --- Cross-notebook: classifier identity -------------------------------
nb01_classifier_type = classifier_summary["metadata"]["classifier_type"]
nb02_classifier_type = attribution_summary["metadata"]["classifier_type"]
assert nb01_classifier_type == nb02_classifier_type, (
    f"Classifier identity drift between notebooks:\n"
    f"  classifier_summary.json:   {nb01_classifier_type!r}\n"
    f"  attribution_summary.json:  {nb02_classifier_type!r}\n"
    f"Re-run Notebook 02 against the classifier Notebook 01 wrote."
)

# --- Subset selection: passing subsets only ----------------------------
gate = classifier_summary["accuracy_gate"]["per_subset"]
passing_subsets = sorted(
    name for name, info in gate.items() if info["verdict"] == "pass"
)
all_subsets = sorted(gate.keys())
non_passing = sorted(set(all_subsets) - set(passing_subsets))

assert passing_subsets, (
    "No subsets passed the Notebook 01 accuracy gate. There is nothing "
    "to interpret here. Revisit Notebook 01's gate threshold or the "
    "classifier configuration."
)

# --- Summary print -----------------------------------------------------
print("Inputs loaded:")
for name, path, extra in [
    ("precommit.json",              precommit_path,            ""),
    ("attribution_scores.parquet",  attribution_scores_path,
        f"({attribution_scores.shape[0]} rows × {attribution_scores.shape[1]} cols)"),
    ("top_attributed_genes.json",   top_attributed_genes_path, ""),
    ("attribution_summary.json",    attribution_summary_path,  ""),
    ("classifier_summary.json",     classifier_summary_path,   ""),
]:
    size_kb = path.stat().st_size / 1e3
    print(f"  {name:<28}  {size_kb:>6.1f} KB  {extra}")
print()
print(f"Attribution method:   {precommit_method}")
print(f"Data source:          {data_scope['source']}, {data_scope['n_stress_classes']} classes")
print(f"Top-N per class:      {attribution_summary['top_attributions']['top_n']}")
print()
print(f"Subsets to process:   {passing_subsets}")
if non_passing:
    print(f"Subsets skipped:      {non_passing}  (did not pass NB01 accuracy gate)")

With the inputs loaded and the cross-checks passed, the rest of the notebook can trust that:

- The precommit's structure is intact and its values agree with Notebook 00's hardcoded facts (data source `GEO`, 8 stress classes).
- The attribution outputs in `attribution_scores`, `top_attributed_genes`, and `attribution_summary` come from a run whose method matches the precommit.
- The list of subsets to process is in `passing_subsets`. Subsets whose classifier didn't pass Notebook 01's accuracy gate are listed under `non_passing` for the EQ#1 writeup but are not analyzed further.

Section 2 builds the reference gene set from Hakkak's supplementary file 1 — the comparator the overlap clause will test against in Section 4.

### 2) Build the reference gene set

The overlap clause in Section 4 needs a comparator — a list of genes that are known, from prior published work, to be stress-responsive in *Arabidopsis*. If the classifier's top-attributed genes overlap meaningfully with that list, the classifier is more likely keying on real stress biology than on dataset artifacts. If they don't overlap, that's evidence of artifact-like behavior.

Your `precommit.json` already records which reference set this notebook uses. The default is Hakkak & Tohidfar (2026), whose Supplementary File 1 lists the 576 differentially expressed genes their meta-analysis identified across drought and salt stress in *Arabidopsis*. The two most notable are *LTP4* (AT5G59310, the most strongly induced) and *GASA9* (AT1G22690, the most strongly repressed) — both used below as anchor checks.

This section loads that file, extracts the AGI codes, and builds the set `reference_agi` that Section 4 will use.

If the precommit's `additional_sources` list is non-empty, those sources would be loaded and unioned in here. The default precommit leaves the list empty, so for most runs Hakkak 2026 is the only source.

In [ ]:
import re

# --- Resolve path and source identity ----------------------------------
primary_source     = precommit["reference_gene_set"]["primary_source"]
additional_sources = precommit["reference_gene_set"]["additional_sources"]

# Currently only one source is supported. The check is structural — if a
# future precommit names a different source, this is where the loader
# registry would be extended.
SUPPORTED_PRIMARY = "Hakkak_Tohidfar_2026"
assert primary_source == SUPPORTED_PRIMARY, (
    f"precommit.json: reference_gene_set.primary_source is "
    f"{primary_source!r}; this notebook currently supports {SUPPORTED_PRIMARY!r} only.\n"
    f"To add support for another source, extend Section 2's loader."
)

hakkak_path = OUTPUT_DIR / "hakkak_2026_supp1.csv"

# --- Defensive load ----------------------------------------------------
# Springer puts a banner row at the top of supplementary CSVs; header=1
# skips it so column names come from row 1 instead of row 0.
try:
    hakkak_raw = pd.read_csv(hakkak_path, header=1)
except FileNotFoundError:
    raise FileNotFoundError(
        f"\nCould not find Hakkak & Tohidfar's Supplementary File 1.\n"
        f"  Expected at: {hakkak_path}\n\n"
        f"In Google Drive, upload `hakkak_2026_supp1.csv` to:\n"
        f"  MyDrive/irilab2026_outputs/r1_q3/\n"
        f"then re-run this cell."
    ) from None

# --- Find the AGI column -----------------------------------------------
# Identify the column whose values look like AGI codes (e.g., AT5G59310).
# This is more durable than hardcoding a column name — Springer's column
# naming varies across submissions, and detecting by content is robust to
# whichever column header was used.
AGI_PATTERN = re.compile(r"^AT[1-5CM]G\d{5}$", re.IGNORECASE)

def _looks_like_agi_column(series):
    sample = series.dropna().astype(str).str.strip().head(20)
    if len(sample) == 0:
        return False
    matches = sample.str.match(AGI_PATTERN).sum()
    return matches >= max(10, int(0.8 * len(sample)))

agi_columns = [c for c in hakkak_raw.columns if _looks_like_agi_column(hakkak_raw[c])]
assert len(agi_columns) == 1, (
    f"Expected exactly one AGI-shaped column in {hakkak_path.name}; "
    f"found {len(agi_columns)}: {agi_columns}.\n"
    f"All columns: {list(hakkak_raw.columns)}\n"
    f"If the file format has changed, update Section 2's column detection."
)
agi_column = agi_columns[0]

# --- Build the reference set -------------------------------------------
hakkak_agi = (
    hakkak_raw[agi_column]
    .dropna()
    .astype(str)
    .str.strip()
    .str.upper()
)
hakkak_agi = set(hakkak_agi[hakkak_agi.str.match(AGI_PATTERN)])

# --- Anchor validation -------------------------------------------------
# LTP4 and GASA9 are named in Hakkak's abstract as the most induced and
# most repressed genes. If either is missing, something has gone wrong
# in the file or the parse.
LTP4  = "AT5G59310"
GASA9 = "AT1G22690"
anchor_missing = [a for a in (LTP4, GASA9) if a not in hakkak_agi]
assert not anchor_missing, (
    f"Anchor AGIs missing from loaded reference set: {anchor_missing}.\n"
    f"Expected both LTP4 ({LTP4}) and GASA9 ({GASA9}) per Hakkak 2026 abstract.\n"
    f"Check that the CSV is Supplementary File 1, not File 2 or 3."
)

# --- Additional sources (placeholder) ----------------------------------
# The default precommit leaves additional_sources empty. If a future
# precommit adds one, extend the loader registry above and union the
# resulting AGI set into reference_agi here.
reference_agi = set(hakkak_agi)
for source_name in additional_sources:
    raise NotImplementedError(
        f"additional_sources contains {source_name!r}, but no loader is "
        f"registered for it. Extend Section 2's loader to add support, "
        f"or remove the entry from precommit.json."
    )

# --- Summary print -----------------------------------------------------
print(f"Primary source:       {primary_source}")
print(f"File:                 {hakkak_path.name}  ({hakkak_path.stat().st_size / 1e3:.1f} KB)")
print(f"AGI column detected:  {agi_column!r}")
print(f"Reference set size:   {len(reference_agi)} unique AGI codes")
print()
print(f"Anchor check:         LTP4 ({LTP4}) and GASA9 ({GASA9}) both present ✓")
print()
print(f"Sample of reference_agi: {sorted(reference_agi)[:5]} ...")
if additional_sources:
    print(f"Additional sources:   {additional_sources}")
else:
    print(f"Additional sources:   none (precommit's list is empty)")

With Section 2 complete, the variable `reference_agi` holds the set of AGI codes against which the overlap clause will be tested. The set is derived purely from the supplementary file — the precommit's `primary_source` field names *which* source was used, but does not contain the gene IDs themselves, keeping the precommit small and the reference data in its canonical location.

Section 3 loads the expression matrix and per-sample metadata. Sections 4 and 5 then apply the two clauses of the artifact-like definition. The reference set you just built feeds Section 4's overlap clause; Section 5's metadata clause uses different inputs (expression values of the top-attributed genes against batch, processing date, and tissue) and doesn't touch `reference_agi`.

### 3) Load the expression matrix and per-sample metadata

Section 4's overlap clause needs to know which features the classifier was trained on — specifically, which are biological probes that can be mapped to AGI codes (and therefore can overlap with the reference set) and which are tissue one-hot columns (which never can).

Section 5's metadata clause needs the expression matrix itself: it takes each (subset, class)'s top-attributed probes and partitions the variance of their expression values against batch, processing date, and tissue identity.

This section loads both — the full AtGenExpress expression matrix and its per-sample metadata — plus the probe-to-AGI mapping that Section 4 will use to translate probe IDs into the AGI codes the reference set is in.

The data volume is modest: roughly 22,810 probes × 248 samples for the expression matrix, 248 rows of metadata, and a ~21,000-entry probe-to-AGI dictionary. Total memory footprint is around 50 MB.

In [ ]:
# --- Load the expression matrix ----------------------------------------
# iri.load_atgenexpress() returns a dict keyed by stress, with each value
# a probes × samples DataFrame for that stress. The probes index is the
# same across stresses (same ATH1 array), so concatenating along axis=1
# yields a unified probes × samples matrix.
per_stress_expr = iri.load_atgenexpress()
expr_matrix = pd.concat(per_stress_expr.values(), axis=1)

# Verify the concat produced a sensible shape — probes on rows, samples
# on columns. The probe count should match the ATH1 array (~22,810).
assert expr_matrix.shape[0] > 20_000, (
    f"expr_matrix has only {expr_matrix.shape[0]} rows; "
    f"expected ~22,810 (ATH1 probe count)."
)
assert expr_matrix.shape[1] > 0, "expr_matrix has no sample columns."

# --- Load per-sample metadata ------------------------------------------
# The metadata DataFrame is indexed by GSM and has six columns:
# stress, tissue, time_h, replicate, gse, last_update_date.
meta = iri.atgenexpress_metadata()

required_meta_cols = {"stress", "tissue", "time_h", "replicate", "gse", "last_update_date"}
missing_cols = required_meta_cols - set(meta.columns)
assert not missing_cols, (
    f"meta is missing expected columns: {sorted(missing_cols)}.\n"
    f"Found columns: {list(meta.columns)}\n"
    f"If `last_update_date` is missing, you may be on an older library "
    f"version. Reinstall irilab2026 from @main and restart the runtime."
)

# --- Probe-to-AGI mapping ----------------------------------------------
# Section 4 uses this to translate the classifier's top-attributed probes
# into AGI codes for the hypergeometric overlap. Building it here so
# Section 4 does not need to re-fetch the GPL198 annotation.
probe_to_agi = iri.probe_to_agi()
assert len(probe_to_agi) > 20_000, (
    f"probe_to_agi has only {len(probe_to_agi)} entries; "
    f"expected ~21,000."
)

# --- Cross-check: attribution_scores samples are in meta ---------------
# Every sample NB02 ran attribution on should exist in the metadata.
# If not, NB01 or NB02 ran against a different cohort than the loader
# now returns — almost certainly a stale Drive output.
attribution_sample_ids = set(attribution_scores["sample_id"])
samples_not_in_meta = attribution_sample_ids - set(meta.index)
assert not samples_not_in_meta, (
    f"{len(samples_not_in_meta)} samples in attribution_scores.parquet "
    f"do not appear in iri.atgenexpress_metadata().\n"
    f"First five: {sorted(samples_not_in_meta)[:5]}\n"
    f"Re-run Notebook 01 and Notebook 02 against the current loader."
)

# --- Cross-check: attribution_scores subsets match passing_subsets -----
# NB02 should have run attribution only on subsets that passed NB01's
# accuracy gate. If the sets diverge, the upstream chain is out of sync.
attribution_subsets = set(attribution_scores["subset"])
expected_subsets    = set(passing_subsets)
assert attribution_subsets == expected_subsets, (
    f"attribution_scores subsets differ from NB01's gate-passing subsets.\n"
    f"  In attribution_scores:    {sorted(attribution_subsets)}\n"
    f"  Gate-passing in NB01:     {sorted(expected_subsets)}\n"
    f"Re-run Notebook 02 against the current Notebook 01 outputs."
)

# --- Summary print -----------------------------------------------------
n_probes_in_map  = sum(1 for p in expr_matrix.index if p in probe_to_agi)
n_probes_no_agi  = expr_matrix.shape[0] - n_probes_in_map

print("Expression matrix:")
print(f"  Shape:                  {expr_matrix.shape[0]} probes × {expr_matrix.shape[1]} samples")
print(f"  Per-stress dataframes:  {len(per_stress_expr)}  ({sorted(per_stress_expr.keys())})")
print()
print("Per-sample metadata:")
print(f"  Shape:                  {meta.shape[0]} samples × {meta.shape[1]} columns")
print(f"  Columns:                {list(meta.columns)}")
print()
print("Probe-to-AGI mapping:")
print(f"  Entries:                {len(probe_to_agi)}")
print(f"  Coverage on expr_matrix: {n_probes_in_map} probes mapped, {n_probes_no_agi} unmapped")
print()
print("Sample counts in passing subsets:")
for subset in passing_subsets:
    n = (attribution_scores["subset"] == subset).sum()
    print(f"  {subset:<20}  {n} samples")

Three new objects are now in scope:

- `expr_matrix` — probes × samples DataFrame holding the full AtGenExpress expression matrix.
- `meta` — GSM-indexed DataFrame with six metadata columns. The three columns the metadata clause cares about are `gse` (batch), `last_update_date` (processing date), and `tissue`. The mapping from precommit-clause variable names to these column names is applied in Section 5 at the point of use, not by renaming the DataFrame here.
- `probe_to_agi` — dictionary mapping ATH1 probe IDs to AGI codes. Section 4 uses this to translate the classifier's top-attributed probes into the identifier space the reference set is in.

Section 4 applies the overlap clause: for each (subset, class), it takes the top-N attributed probes, translates them to AGI codes via `probe_to_agi`, computes the hypergeometric overlap against `reference_agi` from Section 2, and applies the Bonferroni-corrected threshold from the precommit's `overlap_clause`.

### 4) Apply the overlap clause

The overlap clause asks, for each (subset, class) pair, a single statistical question: of the classifier's top-attributed probes for this class, do significantly more overlap with Hakkak's reference set than would be expected by chance?

The test is a right-tail hypergeometric. The universe `N` is the count of AGI codes the classifier could have attributed to — that is, the unique AGIs reachable from the classifier's feature columns via `probe_to_agi`. Tissue one-hot columns and probes without AGI mappings are excluded from `N`, because they can never overlap with an AGI-coded reference set and including them would falsely inflate the universe.

Three other quantities feed the test:

- `K` — the size of the reference set restricted to that same universe (so `K` and `N` are in the same identifier space).
- `n` — the count of unique AGIs the top-attributed probes for this (subset, class) translate to. This will usually be slightly smaller than the raw top-N count from Notebook 02 because some probes drop out (tissue columns; probes without AGI mappings) and the AGI space dedupes when multiple probes target the same gene.
- `k` — the overlap between this (subset, class)'s top AGIs and `K`.

The right-tail p-value is `P(X ≥ k)` under the hypergeometric distribution with parameters `(N, K, n)`. A small p-value means the observed overlap is too large to be chance — evidence the classifier is keying on known stress biology. A large p-value means the overlap is no better than chance — consistent with artifact-like behavior.

Bonferroni correction is applied because the test runs once per class. The precommit pre-computes `corrected_threshold = raw_threshold / n_classes = 0.05 / 8 = 0.00625`. The overlap clause is marked as **failed for this (subset, class)** when `corrected_p_value > corrected_threshold` — that is, when the overlap is *not* significant. (The inverted reading takes a moment to internalize: failure here means "no biological signal detected," which is the artifact-like direction.)

In [ ]:
import math
from scipy.stats import hypergeom

# --- Pull overlap-clause parameters from the precommit -----------------
overlap_clause = artifact_def["overlap_clause"]

assert overlap_clause["test"] == "hypergeometric", (
    f"This section implements hypergeometric only. "
    f"Precommit specifies test={overlap_clause['test']!r}."
)
assert overlap_clause["multiple_testing_correction"] == "bonferroni", (
    f"This section implements Bonferroni only. "
    f"Precommit specifies multiple_testing_correction="
    f"{overlap_clause['multiple_testing_correction']!r}."
)

raw_threshold       = overlap_clause["raw_threshold"]
corrected_threshold = overlap_clause["corrected_threshold"]
n_classes           = artifact_def["n_classes"]

# Belt-and-suspenders: the precommit's corrected threshold should equal
# raw_threshold / n_classes. If not, NB00 wrote inconsistent values.
assert math.isclose(corrected_threshold, raw_threshold / n_classes), (
    f"precommit overlap_clause: corrected_threshold ({corrected_threshold}) "
    f"does not equal raw_threshold ({raw_threshold}) / n_classes ({n_classes})."
)

# --- Build the feature universe in AGI space ---------------------------
# attribution_scores has four bookkeeping columns plus one column per
# feature the classifier saw. Strip the bookkeeping to recover the
# feature columns.
bookkeeping_cols = {"subset", "sample_id", "true_class", "predicted_class"}
feature_cols = [c for c in attribution_scores.columns if c not in bookkeeping_cols]

# Translate feature columns to AGI codes. Anything not in probe_to_agi —
# tissue one-hots and unmapped probes — is excluded from the universe.
universe_agi_full = {probe_to_agi[c] for c in feature_cols if c in probe_to_agi}
N = len(universe_agi_full)
n_tissue_cols     = sum(1 for c in feature_cols if c.startswith("tissue_"))
n_unmapped_probes = len(feature_cols) - len(universe_agi_full.union(
    c for c in feature_cols if c.startswith("tissue_")
))

# Restrict the reference set to the universe — these are the K AGIs the
# classifier *could* have hit. K is what feeds the hypergeometric.
reference_in_universe = reference_agi & universe_agi_full
K = len(reference_in_universe)

# Universe sanity checks.
assert N > 0, "Feature universe is empty after AGI translation."
assert K > 0, (
    f"None of the {len(reference_agi)} reference AGIs are in the "
    f"classifier's feature universe of {N} AGIs. The reference set and "
    f"classifier features may be from incompatible array platforms."
)
assert K < N, (
    f"K ({K}) >= N ({N}). The reference set cannot be larger than the "
    f"universe — something is wrong upstream."
)

# --- Iterate (subset, class), run hypergeometric, record results -------
rows = []
for subset in passing_subsets:
    assert subset in top_attributed_genes, (
        f"Passing subset {subset!r} not present in top_attributed_genes.json. "
        f"Re-run Notebook 02 against the current Notebook 01 outputs."
    )
    for class_name, records in top_attributed_genes[subset].items():
        top_probes = [r["gene_id"] for r in records]
        n_records_raw = len(top_probes)

        # Translate probes → AGIs; tissue one-hots and unmapped probes drop out.
        top_agis = {probe_to_agi[p] for p in top_probes if p in probe_to_agi}
        n = len(top_agis)
        k = len(top_agis & reference_in_universe)

        if n == 0:
            rows.append({
                "subset": subset,
                "class": class_name,
                "n_top_records":   n_records_raw,
                "n_top_agis":      0,
                "n_overlap":       0,
                "expected_overlap": float("nan"),
                "fold_enrichment": float("nan"),
                "p_value":           float("nan"),
                "corrected_p_value": float("nan"),
                "corrected_threshold": corrected_threshold,
                "overlap_failed":  None,
                "note": "No translatable probes in top-N; clause unevaluable.",
            })
            continue

        # Right-tail hypergeometric: P(X >= k) given universe N, references K,
        # draw size n. scipy's sf(k - 1, ...) gives P(X >= k).
        p_value = float(hypergeom.sf(k - 1, N, K, n))
        corrected_p_value = min(p_value * n_classes, 1.0)
        expected_overlap = K * n / N
        fold_enrichment = (k / expected_overlap) if expected_overlap > 0 else float("inf")

        overlap_failed = corrected_p_value > corrected_threshold

        rows.append({
            "subset": subset,
            "class": class_name,
            "n_top_records":      n_records_raw,
            "n_top_agis":         n,
            "n_overlap":          k,
            "expected_overlap":   expected_overlap,
            "fold_enrichment":    fold_enrichment,
            "p_value":            p_value,
            "corrected_p_value":  corrected_p_value,
            "corrected_threshold": corrected_threshold,
            "overlap_failed":     bool(overlap_failed),
            "note":               "",
        })

overlap_results = pd.DataFrame(rows).set_index(["subset", "class"]).sort_index()

# --- Summary print -----------------------------------------------------
print("Feature universe:")
print(f"  Feature columns total:           {len(feature_cols)}")
print(f"  Tissue one-hot columns:          {n_tissue_cols}")
print(f"  Unmapped probes (no AGI):        {n_unmapped_probes}")
print(f"  N (unique AGIs in universe):     {N}")
print()
print("Reference set in universe:")
print(f"  Reference AGIs total:            {len(reference_agi)}")
print(f"  K (reference ∩ universe):        {K}")
print(f"  (lost in restriction:            {len(reference_agi) - K})")
print()
print(f"Bonferroni correction: n_classes = {n_classes},  corrected_threshold = {corrected_threshold:.5f}")
print()
print(f"Per-(subset, class) results:")
display_cols = ["n_top_agis", "n_overlap", "expected_overlap",
                "fold_enrichment", "corrected_p_value", "overlap_failed"]
with pd.option_context("display.float_format", "{:.4g}".format):
    print(overlap_results[display_cols])
print()
fail_count = int(overlap_results["overlap_failed"].fillna(False).sum())
unev_count = int(overlap_results["overlap_failed"].isna().sum())
total      = len(overlap_results)
print(f"Overlap clause failed for {fail_count} of {total} (subset, class) pairs.")
if unev_count:
    print(f"Unevaluable (no translatable probes): {unev_count}.")

The DataFrame `overlap_results` now holds one row per (subset, class), indexed by `(subset, class)`, with these columns:

- `n_top_records` — the raw count of records Notebook 02 wrote into `top_attributed_genes.json` for this (subset, class).
- `n_top_agis` — that count after dropping non-translatable features (tissue columns; probes without AGI mappings) and deduping to unique AGIs. This is `n` in the hypergeometric.
- `n_overlap` — the count of those AGIs that are in the reference set restricted to the universe. This is `k`.
- `expected_overlap` — `K · n / N`, the count expected by chance under the null.
- `fold_enrichment` — `k / expected_overlap`; values much greater than 1 indicate strong enrichment.
- `p_value` — raw right-tail hypergeometric p-value.
- `corrected_p_value` — Bonferroni-corrected (`p_value × n_classes`, clipped to 1.0).
- `corrected_threshold` — the precommit's `0.05 / n_classes`. Identical across all rows.
- `overlap_failed` — `True` when `corrected_p_value > corrected_threshold` (overlap is not significant, consistent with artifact-like behavior); `False` when the overlap is significant; `None` for unevaluable rows.

Section 5 applies the metadata clause next. It works on a different input — the expression values of the top-attributed probes, partitioned against batch, processing date, and tissue — and is independent of `overlap_results`. Section 6 combines the two clauses into per-(subset, class) verdicts.

### 5) Apply the metadata clause

The metadata clause asks a different question than the overlap clause. Where the overlap clause asks whether the classifier's top genes are *biologically* plausible (do they overlap with known stress genes?), the metadata clause asks whether those genes' expression patterns are *technically* confounded — whether their variation across the samples in this subset tracks more strongly with which array batch a sample came from, when it was processed, or which tissue it represents, than would be expected if the classifier were keying on real biology.

For each (subset, class) pair, the procedure is:

1. Take the top-attributed probes for this (subset, class) from Notebook 02.
2. Pull their expression values across all samples in the subset (not just the samples of this class — see note below).
3. For each variable in the precommit's `metadata_clause.variables` list — `batch`, `processing_date`, `tissue` — compute the fraction of expression variance that variable explains, averaged across the top probes. The metric used is η² (eta-squared) from one-way ANOVA: the between-group sum of squares divided by the total sum of squares.
4. Apply the threshold: if any variable explains more than `threshold_pct_variance_explained` (the precommit pins this at 30%), the clause is **failed for this (subset, class)**.

The "samples = full subset, genes = class-specific top-N" choice matters. Restricting samples to the class itself would make the variance partition near-degenerate, since within-class variance is small by construction. The clause is asking: do this class's signature genes show batch/date/tissue structure when looked at across the full subset?

**Zero-variance guard.** Within a subset, some variables may have only one level. In `per_tissue` mode the `tissue` variable is constant within shoot and within root — partitioning against it is degenerate. Other variables can also be constant by chance (e.g., a subset drawing from a single GSE). Any variable with fewer than two unique values within the subset is dropped from that subset's partition, recorded in `dropped_variables`, and excluded from the threshold check. This is the resolution to the open methodological question carried in from the chain-fix handoff.

In [ ]:
import numpy as np

# --- Pull metadata-clause parameters from the precommit ----------------
metadata_clause = artifact_def["metadata_clause"]

assert metadata_clause["test"] == "variance_partitioning", (
    f"This section implements variance partitioning only. "
    f"Precommit specifies test={metadata_clause['test']!r}."
)
assert metadata_clause["fails_when"] == "any_variable_explains_more_than_threshold", (
    f"This section implements the 'any variable above threshold' rule only. "
    f"Precommit specifies fails_when={metadata_clause['fails_when']!r}."
)

variables                       = metadata_clause["variables"]
threshold_pct_variance_explained = metadata_clause["threshold_pct_variance_explained"]

# Map precommit variable names to the metadata DataFrame's column names.
# The metadata DataFrame keeps its loader column names (per Section 3's
# decision); this mapping is applied here, at the read site.
PRECOMMIT_TO_META_COLUMN = {
    "batch":           "gse",
    "processing_date": "last_update_date",
    "tissue":          "tissue",
}
for v in variables:
    assert v in PRECOMMIT_TO_META_COLUMN, (
        f"precommit variables list contains {v!r}, which has no mapping "
        f"to a metadata column. Update PRECOMMIT_TO_META_COLUMN or fix "
        f"the precommit."
    )

# --- Helper: eta-squared averaged across genes -------------------------
def variance_explained_per_gene_avg(expr_sub, group_labels):
    """
    Per-gene eta-squared averaged across genes for one categorical variable.

    expr_sub:     DataFrame indexed by probe ID, columns are sample IDs.
                  Values are expression measurements.
    group_labels: Array-like of group labels, one per sample, in the same
                  order as expr_sub's columns.

    Returns a float in [0, 1] — the average fraction of expression variance
    that the group variable explains, averaged across the probes in expr_sub.
    """
    # Transpose to samples × genes so pandas groupby acts along the row
    # axis (which is what groupby does by default).
    expr_t = expr_sub.T
    group_series = pd.Series(group_labels, index=expr_t.index, name="_group")

    overall_mean = expr_t.mean(axis=0)                              # per-gene
    ss_total     = ((expr_t - overall_mean) ** 2).sum(axis=0)       # per-gene

    group_means = expr_t.groupby(group_series).mean()               # groups × genes
    group_sizes = group_series.value_counts().reindex(group_means.index)

    ss_between = (((group_means - overall_mean) ** 2)
                  .multiply(group_sizes, axis=0)
                  .sum(axis=0))                                     # per-gene

    # Genes with zero total variance contribute zero to the average.
    eta_sq = np.where(ss_total > 0, ss_between / ss_total, 0.0)
    return float(np.mean(eta_sq))

# --- Map subset → list of sample IDs in that subset --------------------
subset_to_samples = (
    attribution_scores.groupby("subset")["sample_id"].apply(list).to_dict()
)

# --- Iterate (subset, class), partition variance, record results -------
metadata_rows = []
for subset in passing_subsets:
    subset_sample_ids = subset_to_samples[subset]
    subset_meta       = meta.loc[subset_sample_ids]

    for class_name, records in top_attributed_genes[subset].items():
        top_probes    = [r["gene_id"] for r in records]
        n_top_records = len(top_probes)

        # Keep only probes that are biological (present in expr_matrix).
        # Tissue one-hot columns and any other non-probe feature drops out.
        biological_probes = [p for p in top_probes if p in expr_matrix.index]
        n_biological      = len(biological_probes)

        if n_biological == 0:
            metadata_rows.append({
                "subset": subset,
                "class":  class_name,
                "n_top_records":     n_top_records,
                "n_top_biological":  0,
                **{f"pct_variance_by_{v}": float("nan") for v in variables},
                "dropped_variables": [],
                "max_pct_variance":  float("nan"),
                "metadata_failed":   None,
                "note": "No biological probes in top-N; clause unevaluable.",
            })
            continue

        expr_sub = expr_matrix.loc[biological_probes, subset_sample_ids]

        # Per-variable variance partition, with zero-variance guard.
        pct_by_var = {}
        dropped    = []
        for precommit_name in variables:
            meta_col   = PRECOMMIT_TO_META_COLUMN[precommit_name]
            group_vals = subset_meta[meta_col].values
            n_levels   = pd.Series(group_vals).nunique()

            if n_levels < 2:
                # Degenerate within this subset — drop with a note.
                pct_by_var[precommit_name] = float("nan")
                dropped.append(precommit_name)
                continue

            eta_sq_avg = variance_explained_per_gene_avg(expr_sub, group_vals)
            pct_by_var[precommit_name] = eta_sq_avg * 100.0  # to percentage

        evaluable_pcts = [v for v in pct_by_var.values() if not pd.isna(v)]

        if not evaluable_pcts:
            metadata_rows.append({
                "subset": subset,
                "class":  class_name,
                "n_top_records":     n_top_records,
                "n_top_biological":  n_biological,
                **{f"pct_variance_by_{v}": pct_by_var[v] for v in variables},
                "dropped_variables": dropped,
                "max_pct_variance":  float("nan"),
                "metadata_failed":   None,
                "note": "All metadata variables degenerate within subset; clause unevaluable.",
            })
            continue

        max_pct         = max(evaluable_pcts)
        metadata_failed = max_pct > threshold_pct_variance_explained

        metadata_rows.append({
            "subset": subset,
            "class":  class_name,
            "n_top_records":     n_top_records,
            "n_top_biological":  n_biological,
            **{f"pct_variance_by_{v}": pct_by_var[v] for v in variables},
            "dropped_variables": dropped,
            "max_pct_variance":  max_pct,
            "metadata_failed":   bool(metadata_failed),
            "note":              "",
        })

metadata_results = pd.DataFrame(metadata_rows).set_index(["subset", "class"]).sort_index()

# --- Summary print -----------------------------------------------------
print(f"Metadata clause variables:        {variables}")
print(f"Threshold (% variance explained): {threshold_pct_variance_explained}")
print()

# Show which variables were dropped in each subset (if any).
print("Per-subset variable applicability:")
for subset in passing_subsets:
    subset_meta_view = meta.loc[subset_to_samples[subset]]
    levels = {
        precommit_name: int(subset_meta_view[meta_col].nunique())
        for precommit_name, meta_col in PRECOMMIT_TO_META_COLUMN.items()
    }
    print(f"  {subset:<20}  levels per variable: {levels}")
print()

pct_cols = [f"pct_variance_by_{v}" for v in variables]
display_cols = (
    ["n_top_biological"] + pct_cols
    + ["max_pct_variance", "dropped_variables", "metadata_failed"]
)
with pd.option_context("display.float_format", "{:.2f}".format,
                       "display.max_colwidth", 40):
    print(metadata_results[display_cols])
print()

fail_count = int(metadata_results["metadata_failed"].fillna(False).sum())
unev_count = int(metadata_results["metadata_failed"].isna().sum())
total      = len(metadata_results)
print(f"Metadata clause failed for {fail_count} of {total} (subset, class) pairs.")
if unev_count:
    print(f"Unevaluable: {unev_count}.")

The DataFrame `metadata_results` now holds one row per (subset, class), indexed by `(subset, class)`, with these columns:

- `n_top_records` — the count of top-attributed records Notebook 02 wrote for this (subset, class).
- `n_top_biological` — that count restricted to probes actually present in the expression matrix (excludes tissue one-hot columns).
- `pct_variance_by_batch`, `pct_variance_by_processing_date`, `pct_variance_by_tissue` — the average η² × 100 for each variable. `NaN` for any variable that was dropped (single level in this subset).
- `dropped_variables` — the list of variables that were dropped from this (subset, class)'s partition because they were degenerate within the subset.
- `max_pct_variance` — the maximum across the evaluable variables.
- `metadata_failed` — `True` when `max_pct_variance > threshold_pct_variance_explained` (a metadata variable explains too much, consistent with artifact-like behavior); `False` when all evaluable variables stay below the threshold; `None` for unevaluable rows.

Section 6 combines `overlap_results` and `metadata_results` into per-(subset, class) verdicts via the precommit's `combination` rule (`AND` — both clauses must fail to flag a class), then rolls per-subset class counts into Strong / Mixed / Low_Overlap verdicts using `lower_cut` and `upper_cut` from the precommit's `rollup` block.

### 6) Combine the clauses and roll up to per-subset verdicts

Sections 4 and 5 produced two independent verdicts per (subset, class):

- `overlap_failed` — the classifier's top genes for this class do *not* overlap significantly with the reference set.
- `metadata_failed` — at least one piece of technical metadata (batch, processing date, or tissue) explains too much of the expression variance of those genes.

This section combines them and rolls them up to a subset-level verdict.

**The combination rule** (precommit's `combination: "AND"`) says both clauses must fail for a class to be flagged as artifact-like. In other words, a class is flagged only when the classifier's top genes are *both* biologically implausible (no significant overlap with known stress genes) *and* technically confounded (track with a metadata variable). If either clause passes — significant overlap, or no technical confound — the class gets a clean bill of health.

That's an intentionally conservative rule. Either signal of biological plausibility is enough to clear the class. The artifact-like flag is reserved for cases where the classifier has neither a biological story nor a clean technical profile.

**The rollup** (precommit's `rollup` block) takes the per-class flags within each subset and converts them to a subset verdict. Count the flagged classes, divide by the evaluable-class count, and apply two thresholds:

| Flagged fraction | Verdict | What it says |
|---|---|---|
| `< 0.25` | **Strong** | Few classes are artifact-like. The classifier is grounded in real biology across most stresses. |
| `0.25 – 0.75` | **Mixed** | Some classes are artifact-like, some aren't. The classifier works in places and breaks in others. |
| `> 0.75` | **Low_Overlap** | Most classes are artifact-like. The classifier is largely keying on dataset artifacts, not biology. |

(The verdict label "Low_Overlap" is a slight misnomer — by the time both clauses fail, the issue is broader than overlap. The name is fixed by the precommit; the meaning is what the table above says.)

**Unevaluable classes** — any class where one or both clauses were marked unevaluable in Sections 4 or 5 — are excluded from the denominator under the precommit's `unevaluable_class_handling: "exclude_from_denominator_with_note"`. The unevaluable class names are recorded in the rollup so the EQ#1 writeup can name them explicitly.

In [ ]:
# --- Pull combination and rollup parameters from the precommit ---------
combination = artifact_def["combination"]
assert combination == "AND", (
    f"This section implements combination='AND' only. "
    f"Precommit specifies combination={combination!r}."
)

rollup = artifact_def["rollup"]
assert rollup["rule"] == "count_based", (
    f"This section implements rule='count_based' only. "
    f"Precommit specifies rule={rollup['rule']!r}."
)
assert rollup["unevaluable_class_handling"] == "exclude_from_denominator_with_note", (
    f"This section implements 'exclude_from_denominator_with_note' only. "
    f"Precommit specifies unevaluable_class_handling="
    f"{rollup['unevaluable_class_handling']!r}."
)

lower_cut       = rollup["lower_cut"]
upper_cut       = rollup["upper_cut"]
verdict_labels  = rollup["verdict_labels"]

assert lower_cut < upper_cut, (
    f"precommit rollup: lower_cut ({lower_cut}) must be < upper_cut ({upper_cut})."
)
expected_label_keys = {"below_lower_cut", "between_cuts", "above_upper_cut"}
assert set(verdict_labels.keys()) == expected_label_keys, (
    f"precommit rollup verdict_labels keys are {sorted(verdict_labels.keys())}; "
    f"expected {sorted(expected_label_keys)}."
)

# --- Build the combined verdicts DataFrame -----------------------------
# overlap_results and metadata_results share the (subset, class) MultiIndex,
# so concatenating along axis=1 lines them up automatically.
overlap_cols  = ["n_top_records", "n_top_agis", "n_overlap", "expected_overlap",
                 "fold_enrichment", "p_value", "corrected_p_value",
                 "corrected_threshold", "overlap_failed"]
metadata_cols = ["n_top_biological", "pct_variance_by_batch",
                 "pct_variance_by_processing_date", "pct_variance_by_tissue",
                 "dropped_variables", "max_pct_variance", "metadata_failed"]

verdicts = pd.concat(
    [overlap_results[overlap_cols], metadata_results[metadata_cols]],
    axis=1,
)
verdicts["overlap_note"]  = overlap_results["note"]
verdicts["metadata_note"] = metadata_results["note"]

# --- Combine clauses to per-class artifact_like flag -------------------
# A class is evaluable iff both clauses are evaluable (neither is None).
# For evaluable classes, artifact_like = overlap_failed AND metadata_failed.
# For unevaluable classes, artifact_like = None.
def _combine_clauses(row):
    o = row["overlap_failed"]
    m = row["metadata_failed"]
    if o is None or m is None or pd.isna(o) or pd.isna(m):
        return None
    return bool(o and m)

verdicts["artifact_like"] = verdicts.apply(_combine_clauses, axis=1)
verdicts["evaluable"]     = verdicts["artifact_like"].notna()

# --- Per-subset rollup -------------------------------------------------
rollup_rows = []
for subset in passing_subsets:
    subset_rows     = verdicts.loc[subset]
    n_classes_total = len(subset_rows)
    evaluable_mask  = subset_rows["evaluable"]
    n_evaluable     = int(evaluable_mask.sum())

    unevaluable_classes = sorted(subset_rows.index[~evaluable_mask].tolist())

    if n_evaluable == 0:
        rollup_rows.append({
            "subset":                 subset,
            "n_classes_total":        n_classes_total,
            "n_evaluable":            0,
            "n_artifact_like":        0,
            "artifact_like_fraction": float("nan"),
            "verdict":                None,
            "unevaluable_classes":    unevaluable_classes,
            "note":                   "All classes unevaluable; subset verdict not assigned.",
        })
        continue

    evaluable_flags = subset_rows.loc[evaluable_mask, "artifact_like"]
    n_artifact_like = int(evaluable_flags.sum())
    fraction        = n_artifact_like / n_evaluable

    # Apply the cuts. Boundary convention: ratio == lower_cut goes to
    # 'between' (Mixed); ratio == upper_cut also goes to 'between'.
    # Mixed is the inclusive bucket; Strong and Low_Overlap are strict.
    if fraction < lower_cut:
        verdict_key = "below_lower_cut"
    elif fraction > upper_cut:
        verdict_key = "above_upper_cut"
    else:
        verdict_key = "between_cuts"
    verdict = verdict_labels[verdict_key]

    note_parts = []
    if unevaluable_classes:
        note_parts.append(
            f"{len(unevaluable_classes)} class(es) excluded from denominator: "
            f"{unevaluable_classes}"
        )
    rollup_rows.append({
        "subset":                 subset,
        "n_classes_total":        n_classes_total,
        "n_evaluable":            n_evaluable,
        "n_artifact_like":        n_artifact_like,
        "artifact_like_fraction": fraction,
        "verdict":                verdict,
        "unevaluable_classes":    unevaluable_classes,
        "note":                   " | ".join(note_parts),
    })

subset_rollup = pd.DataFrame(rollup_rows).set_index("subset")

# --- Summary print -----------------------------------------------------
print("Combination rule:  AND (both clauses must fail to flag a class as artifact-like)")
print(f"Rollup cuts:        lower={lower_cut}, upper={upper_cut}")
print(f"Verdict labels:     {verdict_labels}")
print()

print("Per-(subset, class) verdicts:")
display_cols = ["overlap_failed", "metadata_failed", "artifact_like", "evaluable"]
with pd.option_context("display.max_colwidth", 40):
    print(verdicts[display_cols])
print()

print("Per-subset rollup:")
rollup_display_cols = ["n_classes_total", "n_evaluable", "n_artifact_like",
                       "artifact_like_fraction", "verdict"]
with pd.option_context("display.float_format", "{:.3f}".format):
    print(subset_rollup[rollup_display_cols])
print()

# Summary line per subset, with the note appended if non-empty.
print("Subset verdicts:")
for subset, row in subset_rollup.iterrows():
    verdict_str = row["verdict"] if row["verdict"] is not None else "(no verdict)"
    line = (f"  {subset:<20}  {verdict_str:<14}  "
            f"({row['n_artifact_like']}/{row['n_evaluable']} artifact-like)")
    print(line)
    if row["note"]:
        print(f"      Note: {row['note']}")

Two new objects are now in scope:

- `verdicts` — a (subset, class)-indexed DataFrame holding the full diagnostic record from Sections 4 and 5 plus the combined `artifact_like` flag and the `evaluable` flag. This is the row-level result for the EQ#1 writeup and is what Section 7 will save as `verdicts.parquet`.
- `subset_rollup` — a subset-indexed DataFrame with the rollup counts, the artifact-like fraction, the Strong / Mixed / Low_Overlap verdict, and the list of unevaluable classes (if any). This is the headline result.

Together, `verdicts` and `subset_rollup` are the answer to R1-Q3: when a stress classifier makes its predictions, is it relying on known stress-responsive genes or on patterns that look more like dataset artifacts? The answer is a verdict per subset, with class-level diagnostics that explain *why* that verdict was assigned.

Section 7 writes the two artifacts to disk and produces the EQ#1 deliverable checklist.

### 7) Save the deliverables and finalize EQ#1

This section writes the two deliverable artifacts to disk:

- `verdicts.parquet` — the (subset, class)-level DataFrame from Section 6, with all clause diagnostics, the combined `artifact_like` flag, and the `evaluable` flag. This is the inspectable table that backs every claim in the EQ#1 writeup.
- `comparison_summary.json` — the paper-ready summary. Three top-level keys: `setup` (an echo of the precommit and the upstream attribution settings, so the file is self-contained), `clause_results` (per-(subset, class) overlap and metadata diagnostics, derived from `verdicts`), and `verdicts` (per-(subset, class) artifact-like flags plus the per-subset rollups from `subset_rollup`).

Both files are round-trip verified — the cell writes, reads back, and asserts that structure survives the save/load cycle. Silent corruption (encoding issues, library version drift, etc.) gets caught here, before any downstream consumer reads the files.

A short EQ#1 deliverable checklist prints at the end. The checklist names the four headline numbers per subset (verdict, fraction artifact-like, count evaluable, count unevaluable) and the two file paths a reviewer would open to verify them.

In [ ]:
import json

# --- Resolve output paths ---------------------------------------------
verdicts_path           = OUTPUT_DIR / "verdicts.parquet"
comparison_summary_path = OUTPUT_DIR / "comparison_summary.json"

# --- Write verdicts.parquet -------------------------------------------
# The DataFrame has a MultiIndex (subset, class). Parquet preserves both
# levels through to_parquet/read_parquet without extra arguments.
# `dropped_variables` is a list-of-strings column; parquet handles object
# columns containing lists via pyarrow without intervention.
verdicts.to_parquet(verdicts_path)

# Round-trip verify: row count, column list, and the index level names.
verdicts_reloaded = pd.read_parquet(verdicts_path)
assert len(verdicts_reloaded) == len(verdicts), (
    f"{verdicts_path.name}: row-count mismatch on round-trip "
    f"({len(verdicts_reloaded)} vs {len(verdicts)})."
)
assert list(verdicts_reloaded.columns) == list(verdicts.columns), (
    f"{verdicts_path.name}: column mismatch on round-trip."
)
assert list(verdicts_reloaded.index.names) == ["subset", "class"], (
    f"{verdicts_path.name}: index level names changed on round-trip "
    f"({verdicts_reloaded.index.names})."
)

# --- Build comparison_summary.json ------------------------------------
# clause_results: per-(subset, class), the clause diagnostics in their
# nested form. Keys are stringified (subset, class) tuples → dict form
# does not allow tuple keys in JSON, so the structure is nested by subset.
clause_results = {}
for subset in passing_subsets:
    clause_results[subset] = {}
    for class_name in verdicts.loc[subset].index:
        row = verdicts.loc[(subset, class_name)]
        clause_results[subset][class_name] = {
            "overlap": {
                "n_top_records":       int(row["n_top_records"]),
                "n_top_agis":          int(row["n_top_agis"]) if not pd.isna(row["n_top_agis"]) else None,
                "n_overlap":           int(row["n_overlap"]) if not pd.isna(row["n_overlap"]) else None,
                "expected_overlap":    None if pd.isna(row["expected_overlap"]) else float(row["expected_overlap"]),
                "fold_enrichment":     None if pd.isna(row["fold_enrichment"]) else float(row["fold_enrichment"]),
                "p_value":             None if pd.isna(row["p_value"]) else float(row["p_value"]),
                "corrected_p_value":   None if pd.isna(row["corrected_p_value"]) else float(row["corrected_p_value"]),
                "corrected_threshold": float(row["corrected_threshold"]),
                "failed":              None if row["overlap_failed"] is None or pd.isna(row["overlap_failed"]) else bool(row["overlap_failed"]),
                "note":                row["overlap_note"] or "",
            },
            "metadata": {
                "n_top_biological":              int(row["n_top_biological"]),
                "pct_variance_by_batch":         None if pd.isna(row["pct_variance_by_batch"]) else float(row["pct_variance_by_batch"]),
                "pct_variance_by_processing_date": None if pd.isna(row["pct_variance_by_processing_date"]) else float(row["pct_variance_by_processing_date"]),
                "pct_variance_by_tissue":        None if pd.isna(row["pct_variance_by_tissue"]) else float(row["pct_variance_by_tissue"]),
                "dropped_variables":             list(row["dropped_variables"]),
                "max_pct_variance":              None if pd.isna(row["max_pct_variance"]) else float(row["max_pct_variance"]),
                "threshold_pct_variance_explained": metadata_clause["threshold_pct_variance_explained"],
                "failed":                        None if row["metadata_failed"] is None or pd.isna(row["metadata_failed"]) else bool(row["metadata_failed"]),
                "note":                          row["metadata_note"] or "",
            },
            "combined": {
                "combination":   combination,
                "artifact_like": None if row["artifact_like"] is None or pd.isna(row["artifact_like"]) else bool(row["artifact_like"]),
                "evaluable":     bool(row["evaluable"]),
            },
        }

# verdicts block: per-(subset, class) artifact-like flags plus per-subset rollups.
verdicts_block = {
    "per_class": {
        subset: {
            class_name: None if row["artifact_like"] is None or pd.isna(row["artifact_like"]) else bool(row["artifact_like"])
            for class_name, row in verdicts.loc[subset].iterrows()
        }
        for subset in passing_subsets
    },
    "per_subset": {
        subset: {
            "verdict":                row["verdict"],
            "n_classes_total":        int(row["n_classes_total"]),
            "n_evaluable":            int(row["n_evaluable"]),
            "n_artifact_like":        int(row["n_artifact_like"]),
            "artifact_like_fraction": None if pd.isna(row["artifact_like_fraction"]) else float(row["artifact_like_fraction"]),
            "unevaluable_classes":    list(row["unevaluable_classes"]),
            "note":                   row["note"],
        }
        for subset, row in subset_rollup.iterrows()
    },
}

# setup block: an echo of the inputs so the JSON is self-contained.
setup_block = {
    "precommit":             precommit,
    "attribution_method":    precommit_method,
    "top_n":                 int(attribution_summary["top_attributions"]["top_n"]),
    "passing_subsets":       passing_subsets,
    "non_passing_subsets":   non_passing,
    "reference_source":      primary_source,
    "reference_set_size":    len(reference_agi),
    "reference_in_universe": len(reference_in_universe),
    "feature_universe_size": N,
    "classifier_type":       nb01_classifier_type,
}

comparison_summary = {
    "setup":          setup_block,
    "clause_results": clause_results,
    "verdicts":       verdicts_block,
}

# scipy and numpy can leave non-native scalars in nested structures;
# this default= catches anything we missed during the manual coercion
# above and converts via .item() (numpy scalars) or str (fallback).
def _json_default(obj):
    if hasattr(obj, "item"):
        try:
            return obj.item()
        except (ValueError, TypeError):
            pass
    return str(obj)

with open(comparison_summary_path, "w") as f:
    json.dump(comparison_summary, f, indent=2, default=_json_default)

# Round-trip verify the JSON: top-level keys plus per_subset verdict shape.
with open(comparison_summary_path) as f:
    summary_reloaded = json.load(f)
assert set(summary_reloaded.keys()) == {"setup", "clause_results", "verdicts"}, (
    f"{comparison_summary_path.name}: top-level keys mismatch on round-trip."
)
assert set(summary_reloaded["verdicts"]["per_subset"].keys()) == set(passing_subsets), (
    f"{comparison_summary_path.name}: per_subset keys do not match passing_subsets."
)

# --- File-listing print -----------------------------------------------
print("Saved artifacts:")
for path in [verdicts_path, comparison_summary_path]:
    size_kb = path.stat().st_size / 1e3
    print(f"  {path.name:<28}  {size_kb:>6.1f} KB")
print()

# --- EQ#1 deliverable checklist ---------------------------------------
print("=" * 70)
print("EQ#1 deliverable checklist")
print("=" * 70)
print()
print("Per-subset verdicts:")
for subset, row in subset_rollup.iterrows():
    verdict_str = row["verdict"] if row["verdict"] is not None else "(no verdict)"
    frac_str = (f"{row['artifact_like_fraction']:.3f}"
                if not pd.isna(row["artifact_like_fraction"]) else "n/a")
    print(f"  {subset}")
    print(f"    Verdict:                  {verdict_str}")
    print(f"    Artifact-like fraction:   {frac_str}")
    print(f"    Classes evaluable:        {row['n_evaluable']} of {row['n_classes_total']}")
    if row["unevaluable_classes"]:
        print(f"    Unevaluable classes:      {row['unevaluable_classes']}")
    print()
print("For the EQ#1 writeup:")
print(f"  - Open {verdicts_path.name} to read individual (subset, class) rows.")
print(f"  - Open {comparison_summary_path.name} to copy headline numbers.")
print(f"  - The 'setup' block of the JSON echoes the precommit; quote from it")
print(f"    rather than retyping the methodology choices.")

Notebook 03 is complete. Two artifacts are written and round-trip verified:

- `verdicts.parquet` — the (subset, class) verdict table, ready for inspection.
- `comparison_summary.json` — the paper-ready summary, with the precommit echoed under `setup` so the file is self-contained.

These two files plus the EQ#1 prose are the Week 4 deliverable for R1-Q3. The prose should:

1. State the per-subset verdicts (Strong / Mixed / Low_Overlap) and quote the artifact-like fractions.
2. Reference at least one specific (subset, class) row from `verdicts.parquet` to illustrate the reasoning — typically the class with the most striking diagnostic, or the class whose outcome most influenced the subset verdict.
3. Acknowledge any unevaluable classes by name, with the reason from the `note` field.
4. Name the methodology choices (attribution method, top-N value, reference source, partitioning variables, thresholds) by quoting from the JSON's `setup` block rather than restating them. This keeps the writeup tightly coupled to what the notebook actually computed.

The artifact-like definition the prose is answering — both clauses must fail under AND — is recorded in `setup.precommit.artifact_like_definition`. A reviewer who wants to verify the writeup has everything they need in the two files this section just wrote.